# 5.4 주택 가격 예측 (House Prices) - 주석 설명판

원본 `5_4.ipynb`의 코드에 설명 주석을 단 버전이에요.

**전체 흐름**
1. 데이터 불러오기 / 결측치 확인
2. 타깃(SalePrice) 분포 확인 → 로그 변환
3. 결측치 처리 + 원-핫 인코딩
4. 선형회귀 / 릿지 / 라쏘 학습 → RMSE 비교
5. 회귀 계수 시각화

## 1. 라이브러리 불러오기

In [ ]:
import numpy as np               # 수치 계산 (log1p, sqrt 등)
import pandas as pd              # 표 형태 데이터(DataFrame) 다루기
import seaborn as sns            # 예쁜 통계 그래프
import matplotlib.pyplot as plt  # 기본 그래프
import warnings
warnings.filterwarnings('ignore')  # 경고 메시지 숨기기 (결과 보기 편하게)

## 2. 데이터 불러오기

In [ ]:
# 캐글 House Prices 데이터: 집 1채 = 1행, 집의 특징(피처) 79개 + Id + SalePrice(집값)
df_org = pd.read_csv('./house_train.csv')

# 원본은 보관해 두고 복사본(df)으로 작업 → 실수해도 df_org에서 다시 시작 가능
df = df_org.copy()
df.head()   # 앞 5행 미리보기

## 3. 데이터 크기, 타입, 결측치 확인

In [ ]:
print(df.shape)                 # (행 수, 열 수) → (1460, 81)
print(df.dtypes.value_counts()) # 컬럼 타입별 개수 (object = 문자열 → 나중에 인코딩 필요)

# 컬럼별 결측치(NaN) 개수
isnull_series = df.isnull().sum()

# 결측치가 1개 이상인 컬럼만 골라서 많은 순으로 정렬
print(isnull_series[isnull_series > 0].sort_values(ascending=False))
# → PoolQC, MiscFeature, Alley, Fence 등은 1460개 중 1000개 넘게 비어 있음
#   (거의 다 비어 있는 컬럼은 정보가 없으니 나중에 삭제)

## 4. 타깃값(SalePrice) 분포 확인

In [ ]:
plt.title('Sale Price Histogram')
plt.xticks(rotation=15)                    # x축 숫자가 겹치지 않게 15도 기울이기
sns.histplot(df['SalePrice'], kde=True)    # 히스토그램 + kde(부드러운 분포 곡선)
plt.show()
# → 왼쪽에 몰리고 오른쪽으로 꼬리가 긴 모양 (비싼 집이 소수 있음)
#   이렇게 치우친 분포는 회귀 모델 성능을 떨어뜨림 → 로그 변환으로 펴줌

In [ ]:
plt.title('Log Transformed Sale Price Histogram')

# np.log1p(x) = log(1 + x)
#  - 큰 값은 많이 줄이고 작은 값은 조금 줄여서 분포를 정규분포에 가깝게 만듦
#  - 1을 더하는 이유: x가 0이면 log(0)이 -무한대가 되는 걸 막으려고
log_SalePrice = np.log1p(df['SalePrice'])
sns.histplot(log_SalePrice, kde=True)
plt.show()
# → 좌우가 비교적 대칭인 종 모양으로 바뀜

## 5. 데이터 전처리 (로그 변환, 컬럼 삭제, 결측치 채우기)

In [ ]:
# 로그 변환 전 원래 집값은 따로 보관 (나중에 원래 단위로 비교할 때 사용)
org_SalePrice = df['SalePrice']

# 타깃값을 로그 변환한 값으로 교체 → 이제부터 모델은 '로그 집값'을 예측함
df['SalePrice'] = np.log1p(df['SalePrice'])

# 필요 없는 컬럼 삭제
#  - Id: 그냥 번호라서 집값과 관계 없음
#  - PoolQC, MiscFeature, Alley, Fence, FireplaceQu: 결측치가 너무 많음
df.drop(['Id', 'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu'],
        axis=1, inplace=True)   # axis=1: 열 삭제 / inplace=True: df 자체를 바꿈

# 숫자형 컬럼의 결측치를 각 컬럼의 '평균값'으로 채움
# (문자열 컬럼은 평균을 구할 수 없어서 그대로 NaN으로 남음)
# ※ 최신 pandas에서 에러가 나면 df.fillna(df.mean(numeric_only=True), inplace=True)
df.fillna(df.mean(), inplace=True)

# 아직 결측치가 남은 컬럼 찾기
null_column_count = df.isnull().sum()[df.isnull().sum() > 0]

# 남은 컬럼들의 타입 출력 → 전부 object(문자열)
# → 문자열 결측치는 다음 단계의 원-핫 인코딩이 처리해 줌
print(df.dtypes[null_column_count.index])

## 6. 원-핫 인코딩

In [ ]:
print(df.shape)   # 인코딩 전: (1460, 75)

# pd.get_dummies: 문자열 컬럼을 0/1 컬럼 여러 개로 펼침
#  예) MSZoning 값이 RL, RM, FV ...
#      → MSZoning_RL, MSZoning_RM, MSZoning_FV ... 컬럼이 생기고 해당하면 1, 아니면 0
#  - 모델은 숫자만 계산할 수 있어서 문자열을 이렇게 바꿔줘야 함
#  - NaN은 어떤 컬럼에도 해당하지 않아서 전부 0이 됨 → 결측치 문제도 같이 해결
df_ohe = pd.get_dummies(df)
print(df_ohe.shape)   # 인코딩 후: (1460, 271) → 컬럼이 크게 늘어남

In [ ]:
# 인코딩 후 결측치가 남았는지 다시 확인
null_column_count = df_ohe.isnull().sum()[df_ohe.isnull().sum() > 0]
print(df_ohe.dtypes[null_column_count.index])
# → Series([]) : 비어 있음 = 결측치가 하나도 없음 ✅

## 7. 평가 함수 만들기 (RMSE)

In [ ]:
def get_rmse(model):
    """학습된 모델 하나로 테스트 데이터를 예측하고 RMSE를 계산"""
    pred = model.predict(X_test)              # 테스트 데이터 예측
    mse = mean_squared_error(y_test, pred)    # MSE = 오차 제곱의 평균
    rmse = np.sqrt(mse)                       # RMSE = MSE의 제곱근 (낮을수록 좋음)
    # model.__class__.__name__ : 모델 클래스 이름 문자열 (예: 'Ridge')
    print(model.__class__.__name__, np.round(rmse, 3))
    return rmse

def get_rmses(models):
    """여러 모델의 RMSE를 한 번에 계산해서 리스트로 반환"""
    rmses = []
    for model in models:
        rmse = get_rmse(model)
        rmses.append(rmse)
    return rmses

# ※ 타깃을 로그 변환했기 때문에 여기서 나오는 RMSE는 '로그 집값' 기준
#   → 사실상 RMSLE와 같은 의미 (비율로 얼마나 틀렸는지)

## 8. 선형회귀 / 릿지 / 라쏘 학습 및 비교

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 타깃(정답) = 로그 변환된 집값
y_target = df_ohe['SalePrice']
# 피처(문제) = 집값을 뺀 나머지 컬럼 전부
X_features = df_ohe.drop(['SalePrice'], axis=1, inplace=False)

# 학습용 80%, 테스트용 20%로 나누기 (random_state 고정 → 매번 같은 결과)
X_train, X_test, y_train, y_test = train_test_split(X_features, y_target,
                                                    test_size=0.2, random_state=156)

# 1) 일반 선형회귀: 규제 없음
lr_reg = LinearRegression()
lr_reg.fit(X_train, y_train)

# 2) 릿지(Ridge): L2 규제 → 계수를 전체적으로 작게 만들어 과대적합 방지 (기본 alpha=1)
ridge_reg = Ridge()
ridge_reg.fit(X_train, y_train)

# 3) 라쏘(Lasso): L1 규제 → 덜 중요한 피처의 계수를 아예 0으로 만듦 (기본 alpha=1)
lasso_reg = Lasso()
lasso_reg.fit(X_train, y_train)

models = [lr_reg, ridge_reg, lasso_reg]
get_rmses(models)

# 결과: LinearRegression 0.132 / Ridge 0.128 / Lasso 0.176
#  - 릿지가 가장 좋음
#  - 라쏘가 가장 나쁨 → 기본 alpha=1이 이 데이터엔 너무 강해서
#    중요한 피처까지 계수를 0으로 만들어 버렸기 때문
#    (alpha를 0.001 정도로 줄이면 성능이 좋아짐 → 다음 단계에서 튜닝)

## 9. 회귀 계수 상위/하위 10개 뽑기

In [ ]:
def get_top_bottom_coef(model):
    # 모델의 회귀 계수(coef_)를 피처 이름과 짝지어 Series로 만듦
    coef = pd.Series(model.coef_, index=X_features.columns)

    # 큰 순으로 정렬해서
    coef_high = coef.sort_values(ascending=False).head(10)  # 위에서 10개 = 집값을 가장 올리는 피처
    coef_low = coef.sort_values(ascending=False).tail(10)   # 아래서 10개 = 집값을 가장 내리는 피처
    return coef_high, coef_low

## 10. 모델별 회귀 계수 시각화

In [ ]:
def visualize_coef(models):
    # 가로로 그래프 3개 (모델 3개)
    fig, axs = plt.subplots(figsize=(24, 10), nrows=1, ncols=3)
    fig.tight_layout()   # 그래프끼리 겹치지 않게 간격 조정

    # enumerate: 순서 번호(i_num)와 모델을 같이 꺼냄 → 몇 번째 칸에 그릴지 정하는 데 사용
    for i_num, model in enumerate(models):
        coef_high, coef_low = get_top_bottom_coef(model)

        # 상위 10개 + 하위 10개를 하나로 이어붙임 → 총 20개
        coef_concat = pd.concat([coef_high, coef_low])

        # 그래프 제목 = 모델 이름
        axs[i_num].set_title(model.__class__.__name__ + ' Coeffiecents', size=25)

        # y축 글자(피처 이름)를 그래프 안쪽으로 붙이기 (pad=-120)
        axs[i_num].tick_params(axis='y', direction='in', pad=-120)

        # x축, y축 눈금 글자 크기 키우기
        for label in (axs[i_num].get_xticklabels() + axs[i_num].get_yticklabels()):
            label.set_fontsize(22)

        # x=계수 값, y=피처 이름 → 가로 막대그래프
        sns.barplot(x=coef_concat.values, y=coef_concat.index, ax=axs[i_num])

# ※ 원본은 'model = [...]'(s 없음)으로 만들고 visualize_coef(models)를 호출했어요.
#   위 셀에서 만든 models가 남아 있어서 우연히 동작한 것이라 이름을 맞춰 둠
models = [lr_reg, ridge_reg, lasso_reg]
visualize_coef(models)

# 해석:
#  - 선형회귀: 일부 계수가 아주 크게 튀어 있음 (규제가 없어서)
#  - 릿지: 계수들이 전체적으로 작고 고르게 줄어듦
#  - 라쏘: 대부분 계수가 0이 되고 소수의 피처만 남음 (alpha가 너무 커서)